# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides an example for loading and systematically exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source

The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print metadata summary (use attributes, not subscripting)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Authors: {getattr(dataset.metadata, 'author', 'N/A')}")
print(f"Published: {getattr(dataset.metadata, 'datePublished', 'N/A')}")
print(f"Keywords: {getattr(dataset.metadata, 'keywords', 'N/A')}")

## 2. Data Overview

List available record sets and their `@id` values. Then, list the fields and columns within each record set, always referencing by `@id`.

In [ ]:
# Find all record sets by their '@id'
record_set_objs = [rset for rset in getattr(dataset.metadata, 'recordSet', [])]
if not record_set_objs:
    # mlcroissant 0.8.0+ provides dataset.record_sets()
    if hasattr(dataset, 'record_sets'):
        record_set_objs = list(dataset.record_sets())
    elif hasattr(dataset.metadata, '_record_sets'):
        record_set_objs = dataset.metadata._record_sets
    else:
        record_set_objs = []
print(f"Found {len(record_set_objs)} record sets.")

# Print out the @id and a name/description for each record set
record_set_ids = []
for rset in record_set_objs:
    rid = getattr(rset, '@id', getattr(rset, 'id', None))
    record_set_ids.append(rid)
    print(f"Record set @id: {rid}")
    if hasattr(rset, 'name'):
        print(f"  Name: {rset.name}")
    if hasattr(rset, 'description'):
        print(f"  Description: {rset.description}")
    # List fields and columns by @id
    field_ids = [getattr(f, '@id', getattr(f, 'id', None)) for f in getattr(rset, 'field', [])]
    print(f"  Field @id's: {field_ids}")
    # If columns are present (tabular), show by @id.
    for f in getattr(rset, 'field', []):
        if hasattr(f, 'column'):
            col_ids = [getattr(c, '@id', getattr(c, 'id', None)) for c in (f.column if isinstance(f.column, list) else [f.column])]
            print(f"    Field {getattr(f,'@id',getattr(f,'id',None))} has columns @id's: {col_ids}")
    print()

## 3. Data Extraction

Extract all records for each record set (using their `@id`) into pandas DataFrames. Examine the structure of the first record set as an example.

In [ ]:
# Compose the list of all record set @id's
from collections.abc import Iterable
record_set_ids = [rid for rid in record_set_ids if rid is not None]

# Extract all records to DataFrames, referenced by record set @id
dataframes = {}
for rsid in record_set_ids:
    print(f"Loading records for record set {rsid} ...")
    try:
        records = list(dataset.records(record_set=rsid))
        dataframes[rsid] = pd.DataFrame(records)
        print(f"  # records: {len(records)}; columns: {list(dataframes[rsid].columns)}")
    except Exception as e:
        print(f"  Could not load records for {rsid}: {e}")

# Show columns for first available DataFrame
if dataframes:
    first_rsid = next(iter(dataframes.keys()))
    print(f"\nColumns for record set {first_rsid}:\n", dataframes[first_rsid].columns.tolist())
    display(dataframes[first_rsid].head())
else:
    print('No dataframes loaded!')

## 4. Exploratory Data Analysis (EDA)

Apply data processing to numeric/categorical fields. Filter, normalize, and group data using `@id` field references. Example logic shown for the first record set which contains numeric fields.

In [ ]:
# Identify a numeric field in the first available DataFrame
df = None
numeric_field_id = None
group_field_id = None
selected_rsid = None
for rid, dfi in dataframes.items():
    numeric_candidates = [col for col in dfi.columns if pd.api.types.is_numeric_dtype(dfi[col])]
    if numeric_candidates:
        df = dfi
        selected_rsid = rid
        numeric_field_id = numeric_candidates[0]
        break

if df is not None:
    print(f"Analyzing record set {selected_rsid} and numeric field '{numeric_field_id}'.")
    # If a second numeric field is available, use as group (else look for likely categorical field)
    non_numeric = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if non_numeric:
        group_field_id = non_numeric[0]

    # Drop NaNs in numeric_field for fair comparison
    df_num = df.dropna(subset=[numeric_field_id])

    # Filter: for demonstration, threshold using mean
    threshold = df_num[numeric_field_id].mean()
    filtered_df = df_num[df_num[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.3f}:")
    display(filtered_df.head())

    # Normalization: z-score
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std(ddof=0)
    print(f"Normalized '{numeric_field_id}' (z-score) for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Grouping (if suitable group field exists)
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean '{numeric_field_id}' grouped by '{group_field_id}':")
        display(grouped_df.head())
else:
    print('No suitable numeric field found for EDA!')

## 5. Visualization

Visualize distributions or relationships between data columns using matplotlib. All columns are referenced by their `@id` fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.ylabel(numeric_field_id)
        plt.xlabel(group_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No suitable field for visualization!')

## 6. Conclusion

* This notebook demonstrated how to load, explore, and analyze the FAIR² dataset package using `mlcroissant`, referencing all schema elements by their `@id` fields.
* You may adapt the code to process other record sets or fields by referencing their respective `@id` values (as listed above).
* Further domain-specific analyses can now be performed using the DataFrames created in Section 3.

> For more, see the [mlcroissant documentation](https://mlcommons.github.io/croissant/api/python/).